In [ ]:
!pip install transformers torch numpy matplotlib

# 03 - Transformer Deep Dive

## Overview / 概述

This notebook builds a Transformer from the ground up using NumPy and PyTorch.
We cover self-attention, multi-head attention, causal masking, Rotary Position Embeddings (RoPE),
and positional encoding comparison, plus practical tokenizer and chat-template usage.

**Learning Goals:**
- Implement self-attention from scratch (Q, K, V as matrix multiplications)
- Understand Multi-Head Attention (MHA)
- Apply causal masks for autoregressive generation
- Implement and visualize RoPE
- Compare sinusoidal, RoPE, and ALiBi position encodings
- Use Qwen chat templates and explore BPE tokenizer internals
- Build a mini Transformer decoder block

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Self-Attention from Scratch

Self-attention is the core mechanism of Transformers. Given an input sequence $X \in \mathbb{R}^{n \times d}$
(n tokens, d dimensions), we compute:

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**Intuition:** Each token's Query looks at all other tokens' Keys to decide how much
attention to pay, then aggregates their Values accordingly.

In [ ]:
# === Self-Attention with NumPy (from scratch) ===

def self_attention_numpy(X, W_Q, W_K, W_V):
    """
    Compute self-attention using NumPy.

    Args:
        X: input sequence of shape (n_tokens, d_model)
        W_Q, W_K, W_V: weight matrices of shape (d_model, d_k)

    Returns:
        output: (n_tokens, d_k)
        attn_weights: (n_tokens, n_tokens)
    """
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V

    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    attn_weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
    output = attn_weights @ V

    return output, attn_weights


n_tokens, d_model, d_k = 4, 8, 4
X_np = np.random.randn(n_tokens, d_model)
W_Q_np = np.random.randn(d_model, d_k)
W_K_np = np.random.randn(d_model, d_k)
W_V_np = np.random.randn(d_model, d_k)

out_np, attn_np = self_attention_numpy(X_np, W_Q_np, W_K_np, W_V_np)

print(f"Input shape:  {X_np.shape}")
print(f"Output shape: {out_np.shape}")
print(f"Attention weights shape: {attn_np.shape}")
print(f"\nAttention weights (each row sums to 1):\n{attn_np.round(3)}")
print(f"Row sums: {attn_np.sum(axis=1).round(3)}")

In [ ]:
# === Self-Attention with PyTorch ===

class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
        self.d_k = d_k

    def forward(self, x):
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        out = attn_weights @ V
        return out, attn_weights


batch, d_model, d_k = 2, 8, 4
n_tokens = 4
x_torch = torch.randn(batch, n_tokens, d_model)

sa = SelfAttention(d_model, d_k)
out_torch, attn_torch = sa(x_torch)

print(f"Input shape:  {x_torch.shape}")
print(f"Output shape: {out_torch.shape}")
print(f"Attention weights shape: {attn_torch.shape}")

## 2. Multi-Head Attention (MHA)

Multi-head attention runs several self-attention operations in parallel,
allowing the model to attend to information from different representation subspaces.

**Steps:**
1. Split the embedding dimension into `num_heads` heads
2. Compute scaled dot-product attention for each head in parallel
3. Concatenate all head outputs
4. Project back to `d_model` with a final linear layer

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W_O$$

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention implemented from scratch."""

    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        batch, n_tokens, _ = x.shape

        qkv = self.qkv_proj(x)
        Q, K, V = qkv.chunk(3, dim=-1)

        Q = Q.view(batch, n_tokens, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, n_tokens, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, n_tokens, self.num_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_out = attn_weights @ V

        attn_out = attn_out.transpose(1, 2).contiguous().view(batch, n_tokens, self.d_model)
        out = self.out_proj(attn_out)
        return out, attn_weights


batch, n, d_model, num_heads = 2, 6, 16, 4
x = torch.randn(batch, n, d_model)

mha = MultiHeadAttention(d_model, num_heads)
out, attn = mha(x)

print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention weights: {attn.shape} (batch, heads, tokens, tokens)")
print(f"\nNumber of heads: {num_heads}")
print(f"Dimension per head (d_k): {d_model // num_heads}")

## 3. Causal Mask for Autoregressive Generation

A causal (upper-triangular) mask ensures that token at position `i` can only attend to
positions `0..i`. This is essential for autoregressive language models (decoder-only).

Mask visualization: `1` = allowed to attend, `0` = masked (set to -inf before softmax).

In [ ]:
def create_causal_mask(seq_len):
    """
    Create a causal mask of shape (seq_len, seq_len).
    Lower-triangular = 1 (allowed), upper-triangular = 0 (masked).
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask


seq_len = 8
causal_mask = create_causal_mask(seq_len)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(causal_mask.numpy(), cmap='Blues', vmin=0, vmax=1)
ax.set_title('Causal Mask (1=visible, 0=masked)', fontsize=12)
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, int(causal_mask[i, j]), ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

batch, d_model, num_heads = 2, 16, 4
x = torch.randn(batch, seq_len, d_model)
causal_mask_batch = causal_mask.unsqueeze(0).unsqueeze(0)

mha = MultiHeadAttention(d_model, num_heads)
out_causal, attn_causal = mha(x, mask=causal_mask_batch)

print(f"Attention with causal mask - output shape: {out_causal.shape}")
print(f"\nHead 0 attention pattern (should be lower-triangular):")
print(attn_causal[0, 0].detach().round(3))

## 4. Rotary Position Embedding (RoPE)

RoPE encodes position information by **rotating** the query and key vectors.
Unlike absolute position encodings, RoPE only depends on the *relative* distance between tokens.

**Math:** For a pair of dimensions $(x_{2i}, x_{2i+1})$, apply a 2D rotation:

$$\begin{bmatrix} x_{2i}' \\ x_{2i+1}' \end{bmatrix} =
\begin{bmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\
                \sin(m\theta_i) &  \cos(m\theta_i) \end{bmatrix}
\begin{bmatrix} x_{2i} \\ x_{2i+1} \end{bmatrix}$$

where $m$ is the token position, $\theta_i = \text{base}^{-2i/d}$.

**Key property:** The dot product $q_m \cdot k_n$ depends only on the angle difference
$\theta(m-n)$, which gives RoPE its relative-position behavior.

In [ ]:
def precompute_rope_freqs(dim, max_seq_len=2048, base=10000.0):
    """
    Precompute RoPE rotation frequencies.

    Args:
        dim: head dimension (must be even)
        max_seq_len: maximum sequence length
        base: base for frequency computation (default 10000)

    Returns:
        freqs_cos: (max_seq_len, dim) cosine values
        freqs_sin: (max_seq_len, dim) sine values
    """
    theta = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    positions = torch.arange(max_seq_len).float()
    freqs = torch.outer(positions, theta)

    freqs_cos = freqs.cos().repeat_interleave(2, dim=-1)
    freqs_sin = freqs.sin().repeat_interleave(2, dim=-1)

    return freqs_cos, freqs_sin


def apply_rotary_pos_emb(x, freqs_cos, freqs_sin):
    """
    Apply RoPE to input tensor.

    Args:
        x: (batch, seq_len, dim) input tensor
        freqs_cos, freqs_sin: (seq_len, dim) precomputed frequencies

    Returns:
        rotated tensor of same shape
    """
    seq_len = x.shape[1]
    cos = freqs_cos[:seq_len].unsqueeze(0)
    sin = freqs_sin[:seq_len].unsqueeze(0)

    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]

    rotated_even = x_even * cos[..., 0::2] - x_odd * sin[..., 0::2]
    rotated_odd = x_even * sin[..., 1::2] + x_odd * cos[..., 1::2]

    rotated = torch.stack([rotated_even, rotated_odd], dim=-1).flatten(-2)
    return rotated


batch, seq_len, dim = 1, 16, 8
x = torch.randn(batch, seq_len, dim)

freqs_cos, freqs_sin = precompute_rope_freqs(dim, max_seq_len=seq_len, base=10000.0)
x_rotated = apply_rotary_pos_emb(x, freqs_cos, freqs_sin)

print(f"Original shape:  {x.shape}")
print(f"Rotated shape:   {x_rotated.shape}")
print(f"\nFrequency range: [{freqs_cos.min():.4f}, {freqs_cos.max():.4f}]")

In [ ]:
# === Visualize RoPE rotation angles ===

dim_viz = 64
max_seq_len_viz = 128

freqs_cos_viz, freqs_sin_viz = precompute_rope_freqs(dim_viz, max_seq_len_viz)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

theta = 1.0 / (10000.0 ** (torch.arange(0, dim_viz, 2).float() / dim_viz))
axes[0].plot(range(len(theta)), theta.numpy(), 'b-o', markersize=3)
axes[0].set_title('RoPE Frequencies (by dimension pair)', fontsize=11)
axes[0].set_xlabel('Dimension pair index i')
axes[0].set_ylabel('theta_i (radians)')
axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(freqs_cos_viz.numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[1].set_title(f'RoPE cos(m*theta_i) Heatmap\n({max_seq_len_viz} pos x {dim_viz} dims)', fontsize=11)
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Position m')
plt.colorbar(im, ax=axes[1])

dim_test = 64
base_test = 10000.0
theta_test = 1.0 / (base_test ** (torch.arange(0, dim_test, 2).float() / dim_test))
distances = torch.arange(0, 64)
dot_products = []
for dist in distances:
    dots = (theta_test * dist).cos().sum()
    dot_products.append(dots.item())

axes[2].plot(distances.numpy(), dot_products, 'g-', linewidth=2)
axes[2].set_title('RoPE: Relative-Position Dot Product\n(decays with distance)', fontsize=11)
axes[2].set_xlabel('Relative distance |m - n|')
axes[2].set_ylabel('Dot product (proxy for attention score)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Position Encoding Comparison

| Encoding | Type | Learnable | Max Length | Key Property |
|----------|------|-----------|------------|-------------|
| **Sinusoidal** | Absolute | No | Fixed at training | Uses sin/cos of different frequencies |
| **RoPE** | Relative (via rotation) | No | Extrapolatable* | Encodes relative position in attention dot product |
| **ALiBi** | Relative (via bias) | No | Extrapolatable | Adds a linear bias to attention scores; no embedding needed |
| **Learned** | Absolute | Yes | Fixed at training | Simple but no length generalization |

> *RoPE with NTK-aware scaling can extrapolate to longer sequences.

In [ ]:
# === Implement and compare all three encodings ===

def sinusoidal_position_encoding(seq_len, d_model):
    """Standard sinusoidal position encoding from 'Attention Is All You Need'."""
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(seq_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


def alibi_bias(num_heads, seq_len):
    """
    ALiBi: compute attention bias (additive, not multiplicative).
    Each head gets a different slope. Returns (num_heads, seq_len, seq_len).
    """
    slopes = 2.0 ** (-(2 ** -(math.log2(num_heads) - 3)) * torch.arange(1, num_heads + 1).float())
    positions = torch.arange(seq_len).unsqueeze(1)
    distances = positions - positions.T
    causal_mask = (distances < 0).float()

    biases = []
    for h in range(num_heads):
        bias = -slopes[h] * distances.abs()
        bias = bias * (1 - causal_mask) + causal_mask * float('-inf')
        biases.append(bias)

    return torch.stack(biases, dim=0)


seq_len_pe, d_model_pe, num_heads_pe = 32, 64, 8

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sin_pe = sinusoidal_position_encoding(seq_len_pe, d_model_pe)
im0 = axes[0].imshow(sin_pe.numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[0].set_title('Sinusoidal Position Encoding\n(absolute, fixed)', fontsize=11)
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Position')
plt.colorbar(im0, ax=axes[0])

freqs_cos_comp, _ = precompute_rope_freqs(d_model_pe, seq_len_pe)
im1 = axes[1].imshow(freqs_cos_comp.numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
axes[1].set_title('RoPE (cos component)\n(relative via rotation)', fontsize=11)
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Position')
plt.colorbar(im1, ax=axes[1])

alibi = alibi_bias(num_heads_pe, seq_len_pe)
alibi_viz = alibi[0].clone()
alibi_viz[alibi_viz == float('-inf')] = alibi_viz[alibi_viz != float('-inf')].min() - 1
im2 = axes[2].imshow(alibi_viz.numpy(), aspect='auto', cmap='RdBu_r')
axes[2].set_title('ALiBi bias (head 0)\n(relative, additive)', fontsize=11)
axes[2].set_xlabel('Key position')
axes[2].set_ylabel('Query position')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

print("Position encoding comparison complete.")
print(f"Sinusoidal PE shape: {sin_pe.shape}")
print(f"ALiBi bias shape:   {alibi.shape}")

## 6. Chat Template Demo with Qwen Tokenizer

Tokenizer chat templates format conversational data into the prompt format expected
by the model. We use the Qwen2.5 tokenizer to see how system/user/assistant messages
get transformed into a single token sequence with special tokens.

In [ ]:
from transformers import AutoTokenizer

print("Loading Qwen2.5 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B", trust_remote_code=True)

print(f"BOS token: {tokenizer.bos_token!r} -> id={tokenizer.bos_token_id}")
print(f"EOS token: {tokenizer.eos_token!r} -> id={tokenizer.eos_token_id}")
print(f"PAD token: {tokenizer.pad_token!r} -> id={tokenizer.pad_token_id}")
print(f"Vocab size: {tokenizer.vocab_size}")

print(f"\n--- Chat Template (first 500 chars) ---")
print(tokenizer.chat_template[:500] + "...")

In [ ]:
# === Apply chat template ===

messages = [
    {"role": "system", "content": "You are a helpful programming assistant."},
    {"role": "user", "content": "Write a Python function to compute factorial."},
    {"role": "assistant", "content": "Here is a recursive implementation:\n```python\ndef factorial(n):\n    return 1 if n <= 1 else n * factorial(n-1)\n```"},
    {"role": "user", "content": "Now make it iterative."},
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("=== Formatted Prompt (with special tokens shown) ===")
print(formatted_prompt)
print("\n" + "="*60)

tokenized = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

print(f"\nTokenized shape: {tokenized.shape}")
print(f"Token IDs: {tokenized[0].tolist()}")
print(f"\nDecoded (showing special tokens):")
print(tokenizer.decode(tokenized[0]))

## 7. Tokenizer Internals: BPE Training Demo

Byte-Pair Encoding (BPE) is the most common subword tokenization algorithm.
It starts with characters and iteratively merges the most frequent adjacent pairs.

Here we implement a minimal BPE trainer to understand the core algorithm.

In [ ]:
# === Minimal BPE implementation ===

from collections import Counter


def train_bpe(corpus, num_merges=20, verbose=True):
    """
    Train BPE on a dict of {word: frequency}.

    Args:
        corpus: dict of {word: frequency}
        num_merges: number of merge operations
        verbose: print merge steps

    Returns:
        vocab: dict of {token: id}
        merges: list of ((a,b), merged) operations
        splits: final token splits for each word
    """
    splits = {word: list(chars) + ['</w>'] for word, chars in
              [(w, list(w)) for w in corpus]}

    merges = []
    vocab = set()
    for word in splits:
        vocab.update(splits[word])

    for step in range(num_merges):
        pair_counts = Counter()
        for word, freq in corpus.items():
            tokens = splits[word]
            for i in range(len(tokens) - 1):
                pair_counts[(tokens[i], tokens[i+1])] += freq

        if not pair_counts:
            break

        best_pair = max(pair_counts, key=pair_counts.get)
        new_token = ''.join(best_pair)
        merges.append((best_pair, new_token))
        vocab.add(new_token)

        if verbose:
            print(f"Merge {step+1}: {best_pair} -> '{new_token}' (count={pair_counts[best_pair]})")

        for word in splits:
            tokens = splits[word]
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == best_pair:
                    new_tokens.append(new_token)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            splits[word] = new_tokens

    sorted_vocab = sorted(vocab, key=lambda x: (len(x), x))
    vocab_dict = {tok: i for i, tok in enumerate(sorted_vocab)}

    return vocab_dict, merges, splits


sample_corpus = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
    "lowest": 4,
}

print("Training BPE on sample corpus:")
for w, f in sample_corpus.items():
    print(f"  '{w}': freq={f}")
print()

vocab, merges, final_splits = train_bpe(sample_corpus, num_merges=10)

print(f"\nFinal vocabulary (size={len(vocab)}):")
for tok, tid in sorted(vocab.items(), key=lambda x: x[1]):
    print(f"  {tid:3d}: {tok!r}")

print(f"\nFinal tokenizations:")
for word, tokens in final_splits.items():
    print(f"  '{word}' -> {tokens}")

In [ ]:
# === BPE Encode / Decode + Real Tokenizer Demo ===


def bpe_encode(word, merges):
    """Encode a word using learned BPE merges."""
    tokens = list(word) + ['</w>']
    for (a, b), merged in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


test_words = ["low", "lowest", "wider", "new"]
print("Encoding test words with our BPE:")
for w in test_words:
    tokens = bpe_encode(w, merges)
    print(f"  '{w}' -> {tokens}")


# === Real tokenizer: encode/decode with HuggingFace ===
print("\n--- Real Qwen Tokenizer Encode/Decode ---")
sample_text = "The transformer architecture revolutionized NLP."

encoded = tokenizer.encode(sample_text)
print(f"Text: {sample_text!r}")
print(f"Token IDs: {encoded}")
print(f"Number of tokens: {len(encoded)}")

print(f"\nToken-by-token decode:")
for i, tid in enumerate(encoded):
    print(f"  {tid:5d} -> {tokenizer.decode([tid])!r}")

decoded = tokenizer.decode(encoded)
print(f"\nRound-trip decoded: {decoded!r}")
print(f"Exact match: {decoded == sample_text}")

print(f"\nSpecial token IDs:")
print(f"  <|im_start|>: {tokenizer.convert_tokens_to_ids('<|im_start|>')}")
print(f"  <|im_end|>:   {tokenizer.convert_tokens_to_ids('<|im_end|>')}")

## 8. Build a Mini Transformer Decoder Block

Now we assemble everything into a single decoder block:
- **Multi-Head Self-Attention** (with causal mask + RoPE)
- **Feed-Forward Network** (FFN): two linear layers with activation
- **Layer Norm** + **Residual Connections**

This is the fundamental building block of GPT-style models.

In [ ]:
class RoPEMultiHeadAttention(nn.Module):
    """MHA with RoPE integration."""

    def __init__(self, d_model, num_heads, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        freqs_cos, freqs_sin = precompute_rope_freqs(self.d_k, max_seq_len)
        self.register_buffer('freqs_cos', freqs_cos, persistent=False)
        self.register_buffer('freqs_sin', freqs_sin, persistent=False)

    def forward(self, x, causal_mask=None):
        batch, n_tokens, _ = x.shape

        qkv = self.qkv_proj(x)
        Q, K, V = qkv.chunk(3, dim=-1)

        Q = Q.view(batch, n_tokens, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, n_tokens, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, n_tokens, self.num_heads, self.d_k).transpose(1, 2)

        cos = self.freqs_cos[:n_tokens].view(1, 1, n_tokens, self.d_k)
        sin = self.freqs_sin[:n_tokens].view(1, 1, n_tokens, self.d_k)

        Q_rot = Q * cos + self._rotate_half(Q) * sin
        K_rot = K * cos + self._rotate_half(K) * sin

        scores = Q_rot @ K_rot.transpose(-2, -1) / math.sqrt(self.d_k)

        if causal_mask is not None:
            scores = scores.masked_fill(causal_mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_out = attn_weights @ V

        attn_out = attn_out.transpose(1, 2).contiguous().view(batch, n_tokens, self.d_model)
        return self.out_proj(attn_out)

    @staticmethod
    def _rotate_half(x):
        """Swap and negate half the dimensions for rotation."""
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)


class FeedForward(nn.Module):
    """Standard FFN: Linear -> GELU -> Linear."""

    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff, bias=False)
        self.fc2 = nn.Linear(d_ff, d_model, bias=False)
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class TransformerDecoderBlock(nn.Module):
    """
    A single decoder block:
    1. LayerNorm -> MHA (+RoPE, causal) -> Residual
    2. LayerNorm -> FFN -> Residual
    """

    def __init__(self, d_model, num_heads, d_ff=None, max_seq_len=512):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = RoPEMultiHeadAttention(d_model, num_heads, max_seq_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x, causal_mask=None):
        x = x + self.attn(self.ln1(x), causal_mask)
        x = x + self.ffn(self.ln2(x))
        return x


# --- Test the decoder block ---
batch, seq_len, d_model, num_heads = 2, 16, 64, 4
x = torch.randn(batch, seq_len, d_model)
causal_mask = create_causal_mask(seq_len).unsqueeze(0).unsqueeze(0)

block = TransformerDecoderBlock(d_model, num_heads)
out = block(x, causal_mask)

print(f"Decoder Block Test")
print(f"  Input:  {x.shape}")
print(f"  Output: {out.shape}")
print(f"  Same shape: {x.shape == out.shape}")

total_params = sum(p.numel() for p in block.parameters())
trainable_params = sum(p.numel() for p in block.parameters() if p.requires_grad)
print(f"\n  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Stack multiple blocks
n_layers = 3
mini_transformer = nn.Sequential(*[
    TransformerDecoderBlock(d_model, num_heads) for _ in range(n_layers)
])

out_stacked = mini_transformer(x, causal_mask)
print(f"\n  Stacked ({n_layers} layers) output shape: {out_stacked.shape}")

## TODO: Exercises

1. **Self-Attention**: Modify `self_attention_numpy` to accept an optional mask parameter.
2. **Multi-Head**: Implement MHA where each head has different $W_Q, W_K, W_V$ (not combined projection).
3. **RoPE**: Extend `apply_rotary_pos_emb` to handle batch dimensions properly.
4. **ALiBi**: Integrate ALiBi bias into the `MultiHeadAttention` class.
5. **Decoder Block**: Add dropout to the decoder block (attention dropout + residual dropout).
6. **Tokenizer**: Train BPE on a larger English text and compare with the Qwen tokenizer output.
7. **Analysis**: Plot attention patterns from different heads and interpret what each head learns.

## Summary

In this notebook, we built the core Transformer components from scratch:

| Component | Key Insight |
|-----------|------------|
| **Self-Attention** | Every token attends to every other token via Q*K^T similarity |
| **Multi-Head Attention** | Multiple parallel attention operations capture different relationships |
| **Causal Mask** | Upper-triangular mask ensures autoregressive property (no peeking into future) |
| **RoPE** | Encodes relative position via rotation of Q and K vectors |
| **Position Encodings** | Sinusoidal (absolute), RoPE (relative rotation), ALiBi (additive bias) |
| **Chat Templates** | Format multi-turn conversations with special tokens for instruction models |
| **BPE Tokenization** | Iteratively merges frequent character pairs to build subword vocabulary |
| **Decoder Block** | MHA + FFN with residual connections and layer normalization |

**Next:** Notebook 04 covers streaming chat APIs with FastAPI.

---
*(c) LLM Learning Roadmap - Prerequisites Module*